In [26]:
import os
import sys
from pathlib import Path
import datetime
from typing import List
from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl 
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric import utils
from scipy import sparse
import math

# 添加 MSTNN 模块路径
sys.path.append(str(Path('./MSTNN/MSTNN/training')))
try:
    from Transformer import Encoder
except ImportError:
    print("警告: 无法导入 Transformer 模块，将使用简化版本")
    # 如果导入失败，将在模型定义中使用简化版本

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [27]:
# MSTNN 模型用于市场预测
# 本notebook整合了MSTNN神经网络架构，保持数据特征和输出逻辑不变

In [28]:
# ============ PATHS ============
# DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')
DATA_PATH: Path = Path('./data')

# ============ RETURNS TO SIGNAL CONFIGS ============
MIN_SIGNAL: float = 0.0                         # Minimum value for the daily signal 
MAX_SIGNAL: float = 2.0                         # Maximum value for the daily signal 
SIGNAL_MULTIPLIER: float = 400.0                # Multiplier of the OLS market forward excess returns predictions to signal 

# ============ MSTNN MODEL CONFIGS ============
NUM_STOCKS: int = 1                             # 单股票预测，设为1
WINDOW_SIZE: int = 60                           # 时间窗口大小（用于构建时间序列）
FEATURE_SIZE: int = 13                           # 特征数量（S2, E2, E3, P9, S1, S5, I2, P8, P10, P12, P13, U1, U2）
HIDDEN_SIZE_CNN: int = 10                        # CNN隐藏层大小
SIZE_TRANSF: int = 16                            # Transformer大小
K_NEIGHBORS: int = 1                             # 超边邻居数（单股票设为1）
DROPOUT: float = 0.2                            # Dropout率
NUM_EPOCHS: int = 50                             # 训练轮数
LR: float = 0.001                                # 学习率
L2: float = 1e-2                                 # L2正则化
BATCH_SIZE: int = 32                             # 批次大小

In [29]:
@dataclass
class DatasetOutput:
    X_train : np.ndarray  # (num_samples, window_size, num_stocks, feature_size)
    X_test: np.ndarray
    y_train: np.ndarray
    y_test: np.ndarray
    scaler: StandardScaler

@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float 
    min_signal : float = MIN_SIGNAL
    max_signal : float = MAX_SIGNAL

# 简化的 MSTNN 模型（适配单股票回归任务）
class SimplifiedMSTNN(nn.Module):
    def __init__(self, feature_size, hidden_size_cnn, size_transf, window_size, dropout=0.2):
        super().__init__()
        self.feature_size = feature_size
        self.window_size = window_size
        
        # 简化的CNN特征提取（不使用原始CNNModule，因为需要特定reshape）
        # 使用1D CNN处理时间序列
        self.conv1d_1 = nn.Conv1d(feature_size, hidden_size_cnn, kernel_size=3, padding=1)
        self.conv1d_2 = nn.Conv1d(feature_size, hidden_size_cnn, kernel_size=5, padding=2)
        self.conv1d_pool = nn.AdaptiveAvgPool1d(1)
        
        # 归一化层
        self.norm = nn.LayerNorm(feature_size)
        self.norm1 = nn.LayerNorm(2 * hidden_size_cnn)
        self.norm2 = nn.LayerNorm(size_transf)
        
        # 投影层：将CNN输出维度投影到Transformer输入维度
        self.proj_to_transformer = nn.Linear(2 * hidden_size_cnn, size_transf)
        
        # Transformer编码器（用于时序建模）
        # 创建一个适配的Transformer，修复原始代码的维度问题
        # 原始代码检查 x.shape[2]（batch_size）而不是序列长度，我们需要适配
        self.transformer_base = Encoder(
            input_size=size_transf,
            max_len=window_size,  # 设为实际序列长度
            d_model=size_transf,
            n_head=8,
            n_layers=1,
            drop_prob=dropout,
            device=device,
            window_size=window_size  # 位置编码长度
        )
        
        # MLP层（回归输出）
        self.mlp_1 = nn.Linear(size_transf, size_transf)
        self.act_1 = nn.LeakyReLU(negative_slope=0.1)
        self.mlp_2 = nn.Linear(size_transf, 1)  # 回归输出
        self.dropout = nn.Dropout(dropout)
        
        # 损失函数
        self.loss_function = nn.MSELoss()
    
    def _apply_transformer_layer(self, x, layer, mask):
        """适配的Transformer层，修复设备问题"""
        # x shape: (window_size, 1, batch, d_model) = (60, 1, 32, 16)
        # 获取注意力层
        attention_module = layer.attention
        multi_head_attn = attention_module
        
        # 应用线性变换
        q = multi_head_attn.w_q(x)  # (window_size, 1, batch, d_model)
        k = multi_head_attn.w_k(x)  # (window_size, 1, batch, d_model)
        v = x  # 使用原始x作为v
        
        # 手动实现注意力（修复.cuda()问题）
        # 注意力应该在序列维度（第一个维度）上计算
        # 我们需要重新组织维度: (window_size, 1, batch, d_model) -> (1, batch, window_size, d_model)
        q = q.permute(1, 2, 0, 3)  # (1, batch, window_size, d_model)
        k = k.permute(1, 2, 0, 3)  # (1, batch, window_size, d_model)
        v = v.permute(1, 2, 0, 3)  # (1, batch, window_size, d_model)
        
        d_tensor = k.shape[-1]
        
        # 计算注意力分数: q @ k^T 在序列维度上
        # q: (1, batch, window_size, d_model), k: (1, batch, window_size, d_model)
        k_t = k.transpose(-2, -1)  # (1, batch, d_model, window_size)
        
        # score: (1, batch, window_size, window_size)
        score = torch.matmul(q, k_t) / math.sqrt(d_tensor)
        
        if mask is not None:
            # mask shape: (seq_len, seq_len) = (window_size, window_size)
            # 需要广播到 (1, batch, window_size, window_size)
            mask_expanded = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, window_size, window_size)
            score = score.masked_fill(mask_expanded == 0, -1e9)
        
        att = F.softmax(score, dim=-1)  # (1, batch, window_size, window_size)
        out = torch.matmul(att, v)  # (1, batch, window_size, d_model)
        
        # 转回原始格式: (1, batch, window_size, d_model) -> (window_size, 1, batch, d_model)
        out = out.permute(2, 0, 1, 3)  # (window_size, 1, batch, d_model)
        
        return out
        
    def forward(self, x):
        # x shape: (batch, window_size, num_stocks, feature_size)
        batch_size = x.shape[0]
        x = x.squeeze(2)  # (batch, window_size, feature_size)
        
        # 归一化
        x = self.norm(x)
        
        # 转置为 (batch, feature_size, window_size) 用于1D CNN
        x = x.transpose(1, 2)  # (batch, feature_size, window_size)
        
        # CNN特征提取
        x1 = self.conv1d_1(x)  # (batch, hidden_size_cnn, window_size)
        x2 = self.conv1d_2(x)  # (batch, hidden_size_cnn, window_size)
        
        # 转置回 (batch, window_size, hidden_size_cnn)
        x1 = x1.transpose(1, 2)  # (batch, window_size, hidden_size_cnn)
        x2 = x2.transpose(1, 2)  # (batch, window_size, hidden_size_cnn)
        
        # 拼接
        x = torch.cat([x1, x2], dim=-1)  # (batch, window_size, 2*hidden_size_cnn)
        x = self.norm1(x)
        x = self.dropout(x)
        
        # 投影到Transformer输入维度
        x = self.proj_to_transformer(x)  # (batch, window_size, size_transf)
        
        # Transformer时序建模
        # Transformer期望输入: (batch, window_size, num_stocks, feature_size)
        # 扩展维度: (batch, window_size, size_transf) -> (batch, window_size, 1, size_transf)
        x = x.unsqueeze(2)  # (batch, window_size, 1, size_transf)
        
        # 适配Transformer：原始代码的维度检查有问题，我们需要手动处理位置编码
        # permute: (batch, window_size, 1, size_transf) -> (window_size, 1, batch, size_transf)
        x = x.permute(1, 2, 0, 3)  # (window_size, 1, batch, size_transf)
        
        # 手动添加位置编码（修复原始代码的问题）
        seq_len = x.shape[0]  # 实际序列长度
        pos_enc = self.transformer_base.position_encoding[:seq_len, :]  # (seq_len, d_model)
        pos_enc = pos_enc.unsqueeze(1).unsqueeze(2)  # (seq_len, 1, 1, d_model) 用于广播
        x = x + pos_enc  # (window_size, 1, batch, size_transf)
        
        # 通过Transformer层
        # 确保mask在正确的设备上
        src_mask = self.transformer_base.time_mask[:seq_len, :seq_len].to(x.device)
        
        # 创建一个适配的注意力层来避免硬编码的.cuda()
        # 我们需要手动实现注意力机制，因为原始代码有设备问题
        for layer in self.transformer_base.layers:
            # 手动实现注意力，避免使用原始代码中的.cuda()
            x = self._apply_transformer_layer(x, layer, src_mask)
        
        # 转回原始格式: (window_size, 1, batch, size_transf) -> (batch, window_size, size_transf)
        x = x.permute(2, 0, 1, 3)  # (batch, window_size, 1, size_transf)
        x = x.squeeze(2)  # (batch, window_size, size_transf)
        
        # 取最后一个时间步或平均池化
        x = x[:, -1, :]  # (batch, size_transf) - 使用最后一个时间步
        
        # MLP回归
        x = self.mlp_1(x)  # (batch, size_transf)
        x = self.act_1(x)
        x = self.norm2(x)
        x = self.mlp_2(x)  # (batch, 1)
        
        return x.squeeze(-1)  # (batch,)

In [30]:
ret_signal_params = RetToSignalParameters(
    signal_multiplier= SIGNAL_MULTIPLIER
)

In [31]:
def load_trainset() -> pl.DataFrame:
    """
    Loads and preprocesses the training dataset.

    Returns:
        pl.DataFrame: The preprocessed training DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "train.csv")
        .rename({'market_forward_excess_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
        .head(-10)
    )

def load_testset() -> pl.DataFrame:
    """
    Loads and preprocesses the testing dataset.

    Returns:
        pl.DataFrame: The preprocessed testing DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
    )

def create_example_dataset(df: pl.DataFrame) -> pl.DataFrame:
    """
    Creates new features and cleans a DataFrame.

    Args:
        df (pl.DataFrame): The input Polars DataFrame.

    Returns:
        pl.DataFrame: The DataFrame with new features, selected columns, and no null values.
    """
    vars_to_keep: List[str] = [
        "S2", "E2", "E3", "P9", "S1", "S5", "I2", "P8",
        "P10", "P12", "P13", "U1", "U2"
    ]

    return (
        df.with_columns(
            (pl.col("I2") - pl.col("I1")).alias("U1"),
            (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
        )
        .select(["date_id", "target"] + vars_to_keep)
        .with_columns([
            pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5))
            for col in vars_to_keep
        ])
        .drop_nulls()
    )
    
def join_train_test_dataframes(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    """
    Joins two dataframes by common columns and concatenates them vertically.

    Args:
        train (pl.DataFrame): The training DataFrame.
        test (pl.DataFrame): The testing DataFrame.

    Returns:
        pl.DataFrame: A single DataFrame with vertically stacked data from common columns.
    """
    common_columns: list[str] = [col for col in train.columns if col in test.columns]
    
    return pl.concat([train.select(common_columns), test.select(common_columns)], how="vertical")

def create_time_series_windows(df: pl.DataFrame, features: list[str], window_size: int, scaler: StandardScaler = None) -> tuple:
    """
    将数据转换为时间序列窗口格式
    
    Args:
        df: 输入DataFrame（需要包含date_id和target列）
        features: 特征列表
        window_size: 时间窗口大小
        scaler: 标准化器（如果为None，则创建新的）
    
    Returns:
        X: (num_samples, window_size, num_stocks, feature_size)
        y: (num_samples,)
        scaler: 标准化器
    """
    # 确保按date_id排序
    df_sorted = df.sort('date_id')
    
    # 提取特征和目标
    X_df = df_sorted.select(features)
    y_series = df_sorted.get_column('target')
    
    # 转换为numpy
    X_np = X_df.to_numpy()
    y_np = y_series.to_numpy()
    
    # 标准化
    if scaler is None:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_np)
    else:
        X_scaled = scaler.transform(X_np)
    
    # 创建时间序列窗口
    X_windows = []
    y_windows = []
    
    for i in range(window_size, len(X_scaled)):
        X_window = X_scaled[i-window_size:i]  # (window_size, feature_size)
        # 扩展维度以适配模型: (window_size, feature_size) -> (window_size, num_stocks, feature_size)
        X_window = np.expand_dims(X_window, axis=1)  # (window_size, 1, feature_size)
        X_windows.append(X_window)
        y_windows.append(y_np[i])
    
    if len(X_windows) == 0:
        # 如果没有足够的窗口，返回空数组
        return np.array([]).reshape(0, window_size, 1, len(features)), np.array([]), scaler
    
    return np.array(X_windows), np.array(y_windows), scaler

In [32]:
def convert_ret_to_signal(
    ret_arr: np.ndarray,
    params: RetToSignalParameters
) -> np.ndarray:
    """
    Converts raw model predictions (expected returns) into a trading signal.

    Args:
        ret_arr (np.ndarray): The array of predicted returns.
        params (RetToSignalParameters): Parameters for scaling and clipping the signal.

    Returns:
        np.ndarray: The resulting trading signal, clipped between min and max values.
    """
    return np.clip(
        ret_arr * params.signal_multiplier + 1, params.min_signal, params.max_signal
    )

In [33]:
train: pl.DataFrame = load_trainset()
test: pl.DataFrame = load_testset() 
print(train.tail(3)) 
print(test.head(3))

shape: (3, 98)
┌─────────┬─────┬─────┬─────┬───┬───────────┬─────────────────┬────────────────┬───────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9        ┆ forward_returns ┆ risk_free_rate ┆ target    │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---             ┆ ---            ┆ ---       │
│ i64     ┆ f64 ┆ f64 ┆ f64 ┆   ┆ f64       ┆ f64             ┆ f64            ┆ f64       │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪═════════════════╪════════════════╪═══════════╡
│ 9008    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.530228 ┆ -0.002897       ┆ 0.0001525      ┆ -0.003362 │
│ 9009    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.512769 ┆ -0.027028       ┆ 0.000153       ┆ -0.027493 │
│ 9010    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.015503 ┆ 0.015344        ┆ 0.000153       ┆ 0.014879  │
└─────────┴─────┴─────┴─────┴───┴───────────┴─────────────────┴────────────────┴───────────┘
shape: (3, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬───────────┬─────────────────────┬────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … 

In [34]:
# 合并训练和测试数据以进行统一特征工程
df: pl.DataFrame = join_train_test_dataframes(train, test)
df = create_example_dataset(df=df) 

# 分离训练和测试集
train_df: pl.DataFrame = df.filter(pl.col('date_id').is_in(train.get_column('date_id')))
test_df: pl.DataFrame = df.filter(pl.col('date_id').is_in(test.get_column('date_id')))

FEATURES: list[str] = [col for col in test_df.columns if col not in ['date_id', 'target']]

# 创建时间序列窗口
X_train, y_train, scaler = create_time_series_windows(
    train_df, FEATURES, WINDOW_SIZE, scaler=None
)
X_test, y_test, _ = create_time_series_windows(
    test_df, FEATURES, WINDOW_SIZE, scaler=scaler
)

print(f"训练集形状: X={X_train.shape}, y={y_train.shape}")
print(f"测试集形状: X={X_test.shape}, y={y_test.shape}") 

训练集形状: X=(7450, 60, 1, 13), y=(7450,)
测试集形状: X=(0, 60, 1, 13), y=(0,)


/var/folders/15/z2t8xpvs6xd_55m998pz51480000gn/T/ipykernel_54656/272133195.py:6: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  train_df: pl.DataFrame = df.filter(pl.col('date_id').is_in(train.get_column('date_id')))
/var/folders/15/z2t8xpvs6xd_55m998pz51480000gn/T/ipykernel_54656/272133195.py:7: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  test_df: pl.DataFrame = df.filter(pl.col('date_id').is_in(test.get_column('date_id')))


In [35]:
# 初始化模型
model = SimplifiedMSTNN(
    feature_size=FEATURE_SIZE,
    hidden_size_cnn=HIDDEN_SIZE_CNN,
    size_transf=SIZE_TRANSF,
    window_size=WINDOW_SIZE,
    dropout=DROPOUT
).to(device)

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=L2)

# 转换为torch tensor
X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.FloatTensor(y_train).to(device)
X_test_tensor = torch.FloatTensor(X_test).to(device)
y_test_tensor = torch.FloatTensor(y_test).to(device)

# 创建数据加载器
from torch.utils.data import TensorDataset, DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 训练模型
print("开始训练模型...")
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = model.loss_function(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    
    # 验证
    if (epoch + 1) % 10 == 0 or epoch == 0:
        model.eval()
        with torch.no_grad():
            train_pred = model(X_train_tensor[:1000])  # 只验证前1000个样本，避免内存问题
            train_loss = model.loss_function(train_pred, y_train_tensor[:1000])
            if len(X_test_tensor) > 0:
                test_pred = model(X_test_tensor)
                test_loss = model.loss_function(test_pred, y_test_tensor)
                print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Train Loss: {train_loss.item():.6f}, Test Loss: {test_loss.item():.6f}")
            else:
                print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Train Loss: {train_loss.item():.6f}, Test Loss: N/A (no test data)")

print("训练完成！")

开始训练模型...
Epoch [1/50], Train Loss: 0.000209, Test Loss: N/A (no test data)
Epoch [10/50], Train Loss: 0.000132, Test Loss: N/A (no test data)
Epoch [20/50], Train Loss: 0.000131, Test Loss: N/A (no test data)
Epoch [30/50], Train Loss: 0.000129, Test Loss: N/A (no test data)
Epoch [40/50], Train Loss: 0.000129, Test Loss: N/A (no test data)
Epoch [50/50], Train Loss: 0.000133, Test Loss: N/A (no test data)
训练完成！


In [36]:
# 保存训练好的模型
torch.save({
    'model_state_dict': model.state_dict(),
    'scaler': scaler,
    'features': FEATURES,
    'window_size': WINDOW_SIZE,
}, 'mstnn_model.pth')

def predict_single_row(test_row: pl.DataFrame, historical_data: pl.DataFrame = None, debug=False) -> float:
    """
    对单行数据进行预测
    
    Args:
        test_row: 单行测试数据
        historical_data: 历史数据（用于构建时间窗口，需要已经过特征工程）
        debug: 是否打印调试信息
    
    Returns:
        预测的交易信号
    """
    # 重命名目标列（如果存在）
    if 'lagged_forward_returns' in test_row.columns:
        test_row = test_row.rename({'lagged_forward_returns':'target'})
    
    # 特征工程
    df: pl.DataFrame = create_example_dataset(test_row)
    
    # 如果有历史数据，合并以构建时间窗口
    if historical_data is not None and len(historical_data) > 0:
        # 合并历史数据和当前行
        combined = pl.concat([historical_data, df], how="vertical")
        # 只保留最后WINDOW_SIZE行
        if len(combined) > WINDOW_SIZE:
            combined = combined.tail(WINDOW_SIZE)
    else:
        # 如果没有历史数据，使用当前行重复填充
        combined = pl.concat([df] * WINDOW_SIZE, how="vertical")
        combined = combined.tail(WINDOW_SIZE)
    
    # 确保按date_id排序（如果存在）
    if 'date_id' in combined.columns:
        combined = combined.sort('date_id')
    
    if debug:
        print(f"  组合数据行数: {len(combined)}, date_id范围: {combined['date_id'].min()}-{combined['date_id'].max()}")
    
    # 提取特征
    X_df = combined.select(FEATURES)
    X_np = X_df.to_numpy()
    
    # 标准化
    X_scaled = scaler.transform(X_np)
    
    # 构建窗口: (window_size, feature_size) -> (window_size, num_stocks, feature_size)
    X_window = np.expand_dims(X_scaled, axis=1)  # (window_size, 1, feature_size)
    X_window = np.expand_dims(X_window, axis=0)  # (1, window_size, 1, feature_size)
    
    # 预测
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_window).to(device)
        raw_pred = model(X_tensor).cpu().item()
    
    if debug:
        print(f"  原始预测值: {raw_pred:.8f}")
    
    # 转换为信号
    signal = convert_ret_to_signal(np.array([raw_pred]), ret_signal_params)[0]
    
    if debug:
        print(f"  转换后信号: {signal:.6f}")
    
    return signal

In [37]:
# 模型已训练完成，可以用于预测
print("模型已准备就绪，可以用于预测")

# 测试模型输出是否随输入变化
print("\n测试模型输出变化:")
model.eval()
with torch.no_grad():
    # 测试几个不同的输入
    test_inputs = X_train_tensor[:5]
    test_outputs = model(test_inputs)
    print(f"前5个训练样本的预测值:")
    for i, pred in enumerate(test_outputs.cpu().numpy()):
        print(f"  样本 {i+1}: {pred:.8f}")
    print(f"预测值范围: [{test_outputs.min().item():.8f}, {test_outputs.max().item():.8f}]")
    print(f"预测值标准差: {test_outputs.std().item():.8f}")

模型已准备就绪，可以用于预测

测试模型输出变化:
前5个训练样本的预测值:
  样本 1: 0.00219919
  样本 2: 0.00219919
  样本 3: 0.00219919
  样本 4: 0.00219919
  样本 5: 0.00219919
预测值范围: [0.00219919, 0.00219919]
预测值标准差: 0.00000000


## 生成Submission并计算Sharpe Ratio得分

In [38]:
# 生成完整的submission DataFrame并保存为parquet
print("开始生成预测结果...")

# 获取测试集的所有预测
test_original = pl.read_csv(DATA_PATH / "test.csv")
test_predictions = []

# 使用训练数据作为历史数据（用于构建时间窗口，已经过特征工程）
# 确保历史数据按date_id排序
historical_data = train_df.sort('date_id').tail(WINDOW_SIZE).clone()

print(f"初始历史数据行数: {len(historical_data)}")
print(f"测试集行数: {len(test_original)}")

for idx, row in enumerate(test_original.iter_rows(named=True)):
    # 为每一行创建一个DataFrame
    single_row_df = pl.DataFrame([row])
    
    # 使用predict函数生成预测（前3个打印调试信息）
    debug_mode = (idx < 3)
    if debug_mode:
        print(f"\n预测第 {idx + 1} 行 (date_id={row['date_id']}):")
        print(f"  历史数据行数: {len(historical_data)}, date_id范围: {historical_data['date_id'].min()}-{historical_data['date_id'].max()}")
    
    pred = predict_single_row(single_row_df, historical_data, debug=debug_mode)
    test_predictions.append(pred)
    
    # 更新历史数据（添加当前测试行，用于下一次预测）
    # 将当前测试行添加到历史数据
    current_row = test_original[idx:idx+1]
    current_row = current_row.rename({'lagged_forward_returns':'target'})
    current_row_processed = create_example_dataset(current_row)
    
    # 合并并保持WINDOW_SIZE长度
    historical_data = pl.concat([historical_data, current_row_processed], how="vertical")
    historical_data = historical_data.sort('date_id').tail(WINDOW_SIZE)
    
    if (idx + 1) % 5 == 0 or idx < 3:
        print(f"已处理 {idx + 1}/{len(test_original)} 行, 预测值: {pred:.6f}, 历史数据行数: {len(historical_data)}")

# 创建submission DataFrame
submission_df = pd.DataFrame({
    'date_id': test_original['date_id'].to_list(),
    'row_id': test_original['date_id'].to_list(),
    'prediction': test_predictions
})

# 保存为parquet格式
submission_df.to_parquet('submission.parquet', index=False)

print("\nSubmission DataFrame:")
print(submission_df.head(10))
print(f"\nSubmission shape: {submission_df.shape}")
print(f"\nPrediction statistics:")
print(submission_df['prediction'].describe())
print(f"\n✓ Submission已保存为 submission.parquet")

开始生成预测结果...
初始历史数据行数: 60
测试集行数: 10

预测第 1 行 (date_id=8980):
  历史数据行数: 60, date_id范围: 8961-9010
  组合数据行数: 60, date_id范围: 8962-9010
  原始预测值: 0.00219919
  转换后信号: 1.879677
已处理 1/10 行, 预测值: 1.879677, 历史数据行数: 60

预测第 2 行 (date_id=8981):
  历史数据行数: 60, date_id范围: 8962-9010
  组合数据行数: 60, date_id范围: 8963-9010
  原始预测值: 0.00219919
  转换后信号: 1.879677
已处理 2/10 行, 预测值: 1.879677, 历史数据行数: 60

预测第 3 行 (date_id=8982):
  历史数据行数: 60, date_id范围: 8963-9010
  组合数据行数: 60, date_id范围: 8964-9010
  原始预测值: 0.00219919
  转换后信号: 1.879677
已处理 3/10 行, 预测值: 1.879677, 历史数据行数: 60
已处理 5/10 行, 预测值: 1.879677, 历史数据行数: 60
已处理 10/10 行, 预测值: 1.879677, 历史数据行数: 60

Submission DataFrame:
   date_id  row_id  prediction
0     8980    8980    1.879677
1     8981    8981    1.879677
2     8982    8982    1.879677
3     8983    8983    1.879677
4     8984    8984    1.879677
5     8985    8985    1.879677
6     8986    8986    1.879677
7     8987    8987    1.879677
8     8988    8988    1.879677
9     8989    8989    1.879677

Submission

In [39]:
# 创建solution DataFrame（包含实际的forward_returns和risk_free_rate）
solution_df = test_original.select([
    'date_id', 
    pl.col('lagged_forward_returns').alias('forward_returns'),
    pl.col('lagged_risk_free_rate').alias('risk_free_rate')
]).to_pandas()

solution_df['row_id'] = solution_df['date_id']

print("Solution DataFrame:")
print(solution_df.head())
print(f"\nForward returns 统计:")
print(solution_df['forward_returns'].describe())

Solution DataFrame:
   date_id  forward_returns  risk_free_rate  row_id
0     8980         0.003541        0.000161    8980
1     8981        -0.005964        0.000162    8981
2     8982        -0.007410        0.000160    8982
3     8983         0.005420        0.000160    8983
4     8984         0.008357        0.000159    8984

Forward returns 统计:
count    10.000000
mean      0.001702
std       0.005482
min      -0.007410
25%      -0.001594
50%       0.002674
75%       0.004950
max       0.008357
Name: forward_returns, dtype: float64


In [40]:
# 导入SharpeRatio评分函数并计算得分
from SharpeRatio import score

try:
    sharpe_score = score(
        solution=solution_df, 
        submission=submission_df, 
        row_id_column_name='date_id'
    )
    print(f"{'='*60}")
    print(f"✓ Sharpe Ratio Score: {sharpe_score:.6f}")
    print(f"{'='*60}")
    
    # 显示策略详情
    print(f"\n策略表现:")
    print(f"  - 平均预测信号: {submission_df['prediction'].mean():.4f}")
    print(f"  - 信号标准差: {submission_df['prediction'].std():.4f}")
    print(f"  - 信号范围: [{submission_df['prediction'].min():.4f}, {submission_df['prediction'].max():.4f}]")
    
except Exception as e:
    print(f"❌ 计算Sharpe Ratio时出错: {e}")
    import traceback
    traceback.print_exc()

✓ Sharpe Ratio Score: 2.616359

策略表现:
  - 平均预测信号: 1.8797
  - 信号标准差: 0.0000
  - 信号范围: [1.8797, 1.8797]
